# 08 · Validation against satellite SST
Reanalysis (GLOBAL_MULTIYEAR_PHY_001_030, 1/12°, 0.494 m) vs the **ESA CCI / C3S** satellite climate data record (SST_GLO_SST_L4_REP_OBSERVATIONS_010_024, 0.05°, daily-mean SST at ~20 cm), **1993–2024**. Both use identical zones: 20°E split, 200 m isobath, and a **50 km nearshore band**. Input: `scripts/build_satellite_validation.py`.

The L4 analysis is gap-filled by optimal interpolation and assimilates some of the same observations as the reanalysis, so agreement is not fully independent, but its trends do not depend on the ocean model.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline
import pandas as pd, numpy as np, xarray as xr, matplotlib.pyplot as plt
visualisation.apply_style()
rd = lambda p: pd.read_csv(p, index_col=0, parse_dates=True).asfreq('MS')
sat = rd('../data/processed/sst_satellite_zones_monthly.csv')
rea = rd('../data/processed/thetao_reanalysis_zones_1993_2024.csv')[sat.columns]
maps = xr.open_dataset('../data/processed/trend_maps.nc')
ZONES = list(sat.columns)
LABEL = {z: z.replace('_', ' ').title() for z in ZONES}
sat.index.min(), sat.index.max(), ZONES

## Agreement of monthly values

In [ ]:
rows = {}
for z in ZONES:
    a_s, a_r = preprocessing.anomalies(sat[z]), preprocessing.anomalies(rea[z])
    rows[z] = {'mean_sat': sat[z].mean(), 'mean_rean': rea[z].mean(),
               'bias_rean_minus_sat': (rea[z] - sat[z]).mean(),
               'rmsd': float(np.sqrt(((rea[z] - sat[z]) ** 2).mean())),
               'r_monthly': rea[z].corr(sat[z]), 'r_anomaly': a_r.corr(a_s)}
agree = pd.DataFrame(rows).T
agree.round(3).to_csv('../outputs/tables/08_agreement.csv')
agree.round(3)

## Trends: annual and by season (HAC 95 % CI)

In [ ]:
tr = []
for z in ZONES:
    for src, df in (('satellite', sat), ('reanalysis', rea)):
        a = preprocessing.anomalies(df[z])
        tr.append({'zone': z, 'source': src, 'season': 'ANN', **preprocessing.hac_trend(a)})
        for season, r in preprocessing.seasonal_trends(a).iterrows():
            tr.append({'zone': z, 'source': src, 'season': season, **r.to_dict()})
tr = pd.DataFrame(tr).set_index(['zone', 'season', 'source'])
tr.round(4).to_csv('../outputs/tables/08_trends_satellite_vs_reanalysis.csv')
fmt = lambda r: f"{r.trend_degC_per_decade:+.2f} ± {r.ci95:.2f}{' *' if r.p_value < 0.05 else ''}"
tr.apply(fmt, axis=1).unstack('source').unstack('season').reindex(columns=['ANN', 'DJF', 'MAM', 'JJA', 'SON'], level=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, season in zip(axes, ('ANN', 'DJF')):
    x = np.arange(len(ZONES)); w = 0.38
    for i, (src, c) in enumerate((('reanalysis', '#7c3aed'), ('satellite', '#0f766e'))):
        d = tr.xs((season, src), level=('season', 'source')).loc[ZONES]
        ax.bar(x + (i - 0.5) * w, d['trend_degC_per_decade'], w, yerr=d['ci95'], color=c, label=src,
               capsize=2, error_kw={'lw': 0.7})
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_xticks(x, [LABEL[z].replace(' ', '\n', 1) for z in ZONES], fontsize=7)
    ax.set_title({'ANN': 'Annual', 'DJF': 'Summer (DJF)'}[season] + ' trend 1993–2024')
axes[0].set_ylabel('°C per decade'); axes[0].legend(fontsize=8)
visualisation.source_note(axes[1], 'Data: Copernicus Marine (doi:10.48670/moi-00021; SST_GLO_SST_L4_REP_OBSERVATIONS_010_024)')
fig.tight_layout(); visualisation.save(fig, '08_trend_comparison')

## Is the trend difference significant?
Trend of the monthly difference series (reanalysis − satellite), HAC 95 % CI. A significant positive value means the reanalysis drifts warm relative to the satellite record.

In [ ]:
rows = {}
for z in ZONES:
    a = preprocessing.anomalies(rea[z] - sat[z])
    r, dj = preprocessing.hac_trend(a), preprocessing.seasonal_trends(a).loc['DJF']
    rows[z] = {'ann_diff': r['trend_degC_per_decade'], 'ann_ci': r['ci95'], 'ann_p': r['p_value'],
               'djf_diff': dj['trend_degC_per_decade'], 'djf_ci': dj['ci95'], 'djf_p': dj['p_value']}
diff = pd.DataFrame(rows).T
diff.round(3).to_csv('../outputs/tables/08_trend_difference_rean_minus_sat.csv')
diff.round(3)

## Trend maps
Per-cell OLS trend of monthly anomalies (annual) and of yearly DJF means. Dashed: 50 km from the coast (satellite land mask). Point estimates only — individual cells are noisy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
kw = dict(cmap='RdBu_r', vmin=-0.6, vmax=0.6, add_colorbar=False)
for row, season in enumerate(('annual', 'djf')):
    for col, (src, var) in enumerate((('Reanalysis', f'rean_{season}'), ('Satellite (C3S)', f'sat_{season}'))):
        ax = axes[row, col]; da = maps[var]
        if src == 'Reanalysis': da = da.rename(lat_r='latitude', lon_r='longitude')
        im = da.plot(ax=ax, **kw)
        maps['sat_dist_km'].plot.contour(ax=ax, levels=[50], colors='k', linewidths=0.5, linestyles='--')
        ax.axvline(20, color='k', lw=0.6)
        ax.set(title=f"{src}: {'annual' if season == 'annual' else 'summer (DJF)'}", xlabel='', ylabel='')
        ax.grid(False)
fig.colorbar(im, ax=axes, label='°C per decade', shrink=0.8)
visualisation.save(fig, '08_trend_maps')

## Benguela nearshore summer (DJF) temperature

In [ ]:
def djf(s):
    q = s.resample('QS-DEC'); m = q.mean()[q.count() == 3]
    m = m[m.index.month == 12]; m.index = m.index.year + 1; return m
fig, ax = plt.subplots(figsize=(7, 3))
for src, df, c in (('Reanalysis', rea, '#7c3aed'), ('Satellite', sat, '#0f766e')):
    y = djf(df['benguela_nearshore']); ax.plot(y.index, y, marker='o', ms=3, color=c, label=src)
    b = np.polyfit(y.index, y.values, 1); ax.plot(y.index, np.polyval(b, y.index), color=c, ls='--', lw=1)
ax.set(title='Benguela nearshore (≤50 km) summer mean SST', ylabel='°C', xlabel='Year (Jan–Feb)')
ax.legend(fontsize=8); visualisation.source_note(ax, 'Data: Copernicus Marine'); fig.tight_layout()
visualisation.save(fig, '08_benguela_nearshore_djf')

## Nearshore − offshore ΔT (upwelling proxy), both sources
ΔT = nearshore − offshore, usually negative. A **negative trend** means the nearshore cooled relative to offshore (consistent with stronger upwelling); a positive trend means a weaker thermal contrast.

In [ ]:
rows = []
for r in ('benguela', 'agulhas'):
    for src, df in (('satellite', sat), ('reanalysis', rea)):
        a = preprocessing.anomalies(df[f'{r}_nearshore'] - df[f'{r}_offshore'])
        rows.append({'region': r, 'source': src, 'season': 'ANN', **preprocessing.hac_trend(a)})
        for season, x in preprocessing.seasonal_trends(a).iterrows():
            rows.append({'region': r, 'source': src, 'season': season, **x.to_dict()})
dT = pd.DataFrame(rows).set_index(['region', 'season', 'source'])
dT.round(4).to_csv('../outputs/tables/08_nearshore_offshore_dT_trends.csv')
dT.apply(fmt, axis=1).unstack('source')